# Deep Past Challenge - ByT5 + Ensemble MBR Inference v4

**Target: Score >45** (GeoMean of BLEU × chrF++)

### v4 Fixes (over v3)
1. **FP32 precision**: v3 used float16 which destroyed ByT5 quality (13.7 vs 35.5). Now FP32 like reference.
2. **BF16 autocast**: Only when GPU supports it (A100/H100). T4 → falls back to FP32 automatically.
3. **Matched reference**: 8 beams, 4 beam + 2 sample = 6 MBR candidates (exactly like 35.5 notebook).
4. **Dual GPU**: `device_map="auto"` in FP32 to spread across both T4s.
5. **Fraction fix**: `1/3 → ⅓` works (slash fractions before forbidden char removal).
6. **Time budget**: Auto-skip secondary models if running low on time.

### Requirements
GPU T4x2 | Internet OFF | Runtime < 9 hours

In [ ]:
!pip install -q sacrebleu accelerate

import os, re, json, time, warnings, math
from pathlib import Path
from typing import List
from contextlib import nullcontext

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import sacrebleu

warnings.filterwarnings('ignore')

# BF16 autocast helper (from reference notebook)
def _cuda_bf16_supported():
    if not torch.cuda.is_available():
        return False
    try:
        return bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)())
    except:
        return False

def bf16_autocast_ctx(enabled=True):
    """BF16 autocast if supported, else FP32 (nullcontext). NEVER float16."""
    if enabled and torch.cuda.is_available() and _cuda_bf16_supported():
        return torch.autocast(device_type='cuda', dtype=torch.bfloat16)
    return nullcontext()

USE_BF16 = _cuda_bf16_supported()

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
print(f"BF16 supported: {USE_BF16}")
for i in range(torch.cuda.device_count()):
    n = torch.cuda.get_device_name(i)
    m = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {n} ({m:.1f}GB)")

In [ ]:
# ============================================================
# CONFIGURATION  (v4 — FP32, matched to reference 35.5 notebook)
# ============================================================

# === Time budget ===
WALL_START = time.time()
WALL_LIMIT_S = 8.5 * 3600  # 8.5h hard limit (Kaggle allows 9h)
MAX_MODELS   = 2            # Use at most 2 models

def time_left():
    return WALL_LIMIT_S - (time.time() - WALL_START)

# === Discover ALL available models ===
MODEL_SEARCH_PATHS = [
    # Our fine-tuned model (top priority)
    "/kaggle/input/byt5-akkadian-finetuned/final",
    "/kaggle/input/byt5-akkadian-finetuned",
    # mattiaangeli's model
    "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
    "/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1",
    "/kaggle/input/byt5-akkadian-mbr-v2",
    # manwithacat's models
    "/kaggle/input/models/manwithacat/akkadian-t5-enriched-v2/pytorch/default/1",
    "/kaggle/input/akkadian-t5-enriched-v2/pytorch/default/1",
    "/kaggle/input/akkadian-t5-enriched-v2",
    "/kaggle/input/models/manwithacat/akkadian-byt5-v2/pytorch/default/1",
    "/kaggle/input/akkadian-byt5-v2",
    # Old mt5 model (include for ensemble diversity)
    "/kaggle/input/akkadian-pretrained-and-finetuned-model/best",
    # Local testing
    "../models/byt5-akkadian/final",
]

DISCOVERED_MODELS = []
seen_paths = set()
for p in MODEL_SEARCH_PATHS:
    if Path(p).exists() and p not in seen_paths:
        files = list(Path(p).glob('*.safetensors')) + list(Path(p).glob('*.bin'))
        config = Path(p) / 'config.json'
        if files or config.exists():
            DISCOVERED_MODELS.append(p)
            seen_paths.add(p)

assert len(DISCOVERED_MODELS) > 0, "No models found! Attach at least one model dataset."

if len(DISCOVERED_MODELS) > MAX_MODELS:
    print(f"Found {len(DISCOVERED_MODELS)} models, keeping first {MAX_MODELS}")
    DISCOVERED_MODELS = DISCOVERED_MODELS[:MAX_MODELS]

TEST_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
if not Path(TEST_PATH).exists():
    TEST_PATH = "../data/deep-past-initiative-machine-translation/test.csv"

OUTPUT_DIR = "/kaggle/working/" if Path("/kaggle").exists() else "./output/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# === Generation params (matched to reference 35.5 notebook) ===
MAX_LENGTH = 384
NUM_BEAMS = 8                  # same as reference
MAX_NEW_TOKENS = 256
LENGTH_PENALTY = 1.3
REPETITION_PENALTY = 1.2

# === Batch size ===
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
if N_GPUS > 0:
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    if vram >= 30:
        BATCH_SIZE = 8
    elif vram >= 14:
        BATCH_SIZE = 4  # reference uses 4 on T4
    else:
        BATCH_SIZE = 2
else:
    BATCH_SIZE = 1

NUM_WORKERS = 4  # reference uses 4

# === MBR config (matched to reference: 4 beam + 2 sample = 6) ===
USE_MBR = True
MBR_NUM_BEAM_CANDS = 4         # reference: 4
MBR_SAMPLE_CONFIGS = [
    (0.7, 0.90, 2),            # reference: temp=0.7, top_p=0.9, 2 samples
]
MBR_UTILITY = 'chrf'           # reference uses chrF++

PREFIX = "translate Akkadian to English: "

total_per_model = MBR_NUM_BEAM_CANDS + sum(c[2] for c in MBR_SAMPLE_CONFIGS)
total_all = total_per_model * len(DISCOVERED_MODELS)

print(f"\nDiscovered {len(DISCOVERED_MODELS)} model(s):")
for i, m in enumerate(DISCOVERED_MODELS):
    print(f"  [{i}] {m}")
print(f"\nMBR: {MBR_NUM_BEAM_CANDS} beam + {sum(c[2] for c in MBR_SAMPLE_CONFIGS)} sample = {total_per_model}/model × {len(DISCOVERED_MODELS)} = {total_all} total")
print(f"Precision: FP32 (BF16 autocast: {USE_BF16})")
print(f"Batch: {BATCH_SIZE}, Beams: {NUM_BEAMS}, GPUs: {N_GPUS}")

In [ ]:
# ============================================================
# PREPROCESSING - Minimal (matched to model training)
# ============================================================
# Critical insight: the high-scoring model uses MINIMAL preprocessing.
# Only gap normalization. No determinative conversion, no H replacement.
# The model learns to handle raw ATF notation internally.

_RE_BIG_GAP = re.compile(r'(\.{3,}|\u2026+|\u2026\u2026)')
_RE_SMALL_GAP = re.compile(r'(xx+|\s+x\s+)')

def preprocess(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = _RE_BIG_GAP.sub('<big_gap>', text)
    text = _RE_SMALL_GAP.sub('<gap>', text)
    return text

def preprocess_batch(texts):
    s = pd.Series(texts).fillna('').astype(str)
    s = s.str.replace(_RE_BIG_GAP, '<big_gap>', regex=True)
    s = s.str.replace(_RE_SMALL_GAP, '<gap>', regex=True)
    return s.tolist()

print("Preprocessing defined.")

In [ ]:
# ============================================================
# POSTPROCESSING - Aggressive (matched to 35.5 approach)
# ============================================================
# BUG FIX v3: Fraction replacement MUST happen BEFORE forbidden char
# removal, because '/' is in _FORBIDDEN and gets stripped first,
# turning "1/3" into "13".

_PP_PATTERNS = {
    'gap': re.compile(r'(\[x\]|\(x\)|\bx\b)', re.I),
    'big_gap': re.compile(r'(\.{3,}|\u2026|\[\.+\])'),
    'annotations': re.compile(r'\((fem|plur|pl|sing|singular|plural|\?|!)\.*\s*\w*\)', re.I),
    'repeated_words': re.compile(r'\b(\w+)(?:\s+\1\b)+'),
    'whitespace': re.compile(r'\s+'),
    'punct_space': re.compile(r'\s+([.,:])'),
    'repeated_punct': re.compile(r'([.,])\1+'),
}

_SUBSCRIPT_TRANS = str.maketrans(
    '\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089',
    '0123456789'
)
_SPECIAL_TRANS = str.maketrans('\u1e2b\u1e2a', 'hH')

# Forbidden chars to remove (gaps and fractions are protected first)
_FORBIDDEN = '!?()"\u2014\u2014<>\u2308\u2309\u230a\u230b[]+\u02be/;'
_FORBIDDEN_TRANS = str.maketrans('', '', _FORBIDDEN)

_NGRAM_PATS = []
for n in range(4, 1, -1):
    p = r'\b((?:\w+\s+){' + str(n-1) + r'}\w+)(?:\s+\1\b)+'
    _NGRAM_PATS.append(re.compile(p))


def postprocess_batch(translations):
    """Postprocess translations with vectorized pandas operations."""
    s = pd.Series(translations).fillna('').astype(str)
    
    # Basic cleaning
    s = s.str.translate(_SPECIAL_TRANS)
    s = s.str.translate(_SUBSCRIPT_TRANS)
    s = s.str.replace(_PP_PATTERNS['whitespace'], ' ', regex=True).str.strip()
    
    # Gap normalization
    s = s.str.replace(_PP_PATTERNS['gap'], '<gap>', regex=True)
    s = s.str.replace(_PP_PATTERNS['big_gap'], '<big_gap>', regex=True)
    s = s.str.replace('<gap> <gap>', '<big_gap>', regex=False)
    s = s.str.replace('<big_gap> <big_gap>', '<big_gap>', regex=False)
    
    # Remove annotations like (fem), (plur), etc.
    s = s.str.replace(_PP_PATTERNS['annotations'], '', regex=True)

    # ── FRACTIONS FIRST ── before forbidden char removal (which strips '/')
    # Slash fractions (must be before '/' removal!)
    s = s.str.replace(r'\b1/3\b', '⅓', regex=True)
    s = s.str.replace(r'\b2/3\b', '⅔', regex=True)
    s = s.str.replace(r'\b1/6\b', '⅙', regex=True)
    s = s.str.replace(r'\b5/6\b', '⅚', regex=True)
    s = s.str.replace(r'\b1/2\b', '½', regex=True)
    s = s.str.replace(r'\b1/4\b', '¼', regex=True)
    s = s.str.replace(r'\b3/4\b', '¾', regex=True)
    # Decimal fractions (specific 0.X first, then generic N.X)
    s = s.str.replace(r'\b0\.5\b',    '½',        regex=True)
    s = s.str.replace(r'(\d+)\.5\b',  '\\g<1>½',  regex=True)
    s = s.str.replace(r'\b0\.25\b',   '¼',        regex=True)
    s = s.str.replace(r'(\d+)\.25\b', '\\g<1>¼',  regex=True)
    s = s.str.replace(r'\b0\.75\b',   '¾',        regex=True)
    s = s.str.replace(r'(\d+)\.75\b', '\\g<1>¾',  regex=True)

    # ── NOW remove forbidden chars (with gap protection) ──
    s = s.str.replace('<gap>', '\x00GAP\x00', regex=False)
    s = s.str.replace('<big_gap>', '\x00BIG\x00', regex=False)
    s = s.str.translate(_FORBIDDEN_TRANS)
    s = s.str.replace('\x00GAP\x00', ' <gap> ', regex=False)
    s = s.str.replace('\x00BIG\x00', ' <big_gap> ', regex=False)
    
    # Repeated words and n-grams
    s = s.str.replace(_PP_PATTERNS['repeated_words'], r'\1', regex=True)
    for pat in _NGRAM_PATS:
        s = s.str.replace(pat, r'\1', regex=True)
    
    # Fix punctuation
    s = s.str.replace(_PP_PATTERNS['punct_space'], r'\1', regex=True)
    s = s.str.replace(_PP_PATTERNS['repeated_punct'], r'\1', regex=True)
    s = s.str.replace(_PP_PATTERNS['whitespace'], ' ', regex=True)
    s = s.str.strip().str.strip('-').str.strip()
    
    return s.tolist()


# Self-test — verify fraction bug is fixed
_test = postprocess_batch([
    "Seal of Mannum-balum-Aššur son of Ṣill-Adad.",
    "He paid 1.5 minas of silver.",
    "0.5 shekel and 1/3 mina.",
    "2/3 mina and 1/6 shekel.",
])
assert '⅓' in _test[2], f"Fraction 1/3 bug NOT fixed: {_test[2]}"
assert '⅔' in _test[3], f"Fraction 2/3 bug NOT fixed: {_test[3]}"
assert '⅙' in _test[3], f"Fraction 1/6 bug NOT fixed: {_test[3]}"
assert len(_test) == 4 and all(len(t) > 0 for t in _test), f"Postprocess self-test FAILED: {_test}"
print(f"Postprocessing defined and self-tested OK:")
for t in _test:
    print(f"  → {t}")

In [ ]:
# ============================================================
# MBR DECODING - chrF++ (fast) or Geo-Mean utility
# ============================================================
# v3: Default to chrF++ only for speed. Geo-mean computes both
# BLEU and chrF++ per pair → 2x slower. chrF++ alone is nearly
# as good for MBR selection and much faster.

_CHRFPP = sacrebleu.metrics.CHRF(word_order=2)
_BLEU = sacrebleu.metrics.BLEU(effective_order=True)

def _sim_chrf(a, b):
    """chrF++ similarity between two strings."""
    a = (a or '').strip()
    b = (b or '').strip()
    if not a or not b:
        return 0.0
    return float(_CHRFPP.sentence_score(a, [b]).score)

def _sim_geomean(a, b):
    """Geometric mean of BLEU and chrF++ — matches competition metric."""
    a = (a or '').strip()
    b = (b or '').strip()
    if not a or not b:
        return 0.0
    bleu = float(_BLEU.sentence_score(a, [b]).score)
    chrf = float(_CHRFPP.sentence_score(a, [b]).score)
    return math.sqrt(max(bleu, 0.1) * max(chrf, 0.1))


def mbr_pick(cands, utility='chrf'):
    """Select best candidate using MBR with specified utility function."""
    # Deduplicate
    seen = set()
    unique = []
    for c in cands:
        c = str(c).strip()
        if c and c not in seen:
            unique.append(c)
            seen.add(c)
    
    if len(unique) == 0: return ''
    if len(unique) == 1: return unique[0]
    
    sim_fn = _sim_geomean if utility == 'geomean' else _sim_chrf
    
    n = len(unique)
    scores = [0.0] * n
    for i in range(n):
        for j in range(n):
            if i != j:
                scores[i] += sim_fn(unique[i], unique[j])
        scores[i] /= max(1, n - 1)
    
    best_i = max(range(n), key=lambda i: scores[i])
    return unique[best_i]


# Self-test
_test_cands = [
    "Seal of Mannum-balum-Aššur, son of Ṣilli-Adad.",
    "Seal of Mannum-balum-Aššur son of Ṣill-Adad.",
    "The seal of Mannum-balum-Aššur, son of Ṣilli-Adad.",
    "Mannum-balum-Aššur son of Ṣill-Adad seal.",
]
_pick = mbr_pick(_test_cands, utility='chrf')
print(f"MBR defined (default: chrF++). Self-test: '{_pick}'")

In [ ]:
# ============================================================
# LOAD MODEL(S) - FP32, dual GPU with device_map="auto"
# ============================================================
# CRITICAL: ByT5 needs FP32. float16 destroys quality (35.5 → 13.7).
# BF16 autocast is used during generation only if GPU supports it.
import gc
from transformers import AutoConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Inspect discovered models
model_configs = []
for model_path in DISCOVERED_MODELS:
    try:
        cfg = AutoConfig.from_pretrained(model_path)
        model_type = cfg.model_type
        model_configs.append({
            'path': model_path,
            'type': model_type,
            'prefix': PREFIX,
            'name': Path(model_path).parent.name + '/' + Path(model_path).name,
        })
        print(f"  OK {model_configs[-1]['name']} ({model_type})")
    except Exception as e:
        print(f"  SKIP {model_path}: {e}")

def load_model(path):
    """Load model in FP32. Use device_map='auto' for multi-GPU."""
    tok = AutoTokenizer.from_pretrained(path)
    if N_GPUS >= 2:
        # device_map="auto" distributes layers across GPUs in FP32
        mdl = AutoModelForSeq2SeqLM.from_pretrained(path, device_map="auto")
    else:
        mdl = AutoModelForSeq2SeqLM.from_pretrained(path)
        if torch.cuda.is_available():
            mdl = mdl.to(device)
    mdl.eval()
    
    # Try BetterTransformer for speed (only works without device_map)
    if not hasattr(mdl, 'hf_device_map'):
        try:
            from optimum.bettertransformer import BetterTransformer
            mdl = BetterTransformer.transform(mdl)
            print('  BetterTransformer applied')
        except:
            pass
    
    return tok, mdl

def free_model(mdl):
    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Load primary model
print(f"\nLoading primary model: {model_configs[0]['path']}")
primary_tokenizer, primary_model = load_model(model_configs[0]['path'])

if hasattr(primary_model, 'hf_device_map'):
    print(f"  device_map: {primary_model.hf_device_map}")
else:
    print(f"  device: {next(primary_model.parameters()).device}")

n_params = sum(p.numel() for p in primary_model.parameters())
dtype = next(primary_model.parameters()).dtype
print(f"  Loaded: {n_params/1e6:.0f}M params, dtype={dtype}")

if N_GPUS >= 2:
    for i in range(N_GPUS):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_mem / 1e9
        print(f"  GPU {i}: {alloc:.1f}/{total:.1f} GB used")

try:
    torch.set_float32_matmul_precision('high')
except:
    pass

print(f"\nTotal models for ensemble: {len(model_configs)}")

In [ ]:
# ============================================================
# LOAD & PREPROCESS TEST DATA
# ============================================================

test_df = pd.read_csv(TEST_PATH)
print(f"Test: {len(test_df)} samples")

# Minimal preprocessing
raw_texts = test_df['transliteration'].tolist()
processed = preprocess_batch(raw_texts)
test_df['input_text'] = [PREFIX + t for t in processed]
test_df['tlen'] = test_df['input_text'].str.split().str.len()

# Sort by length for efficient batching (short first = faster throughput)
test_df = test_df.sort_values('tlen').reset_index(drop=True)

print(f"\nLength distribution:")
print(f"  Mean: {test_df['tlen'].mean():.0f} words")
print(f"  Median: {test_df['tlen'].median():.0f} words")
print(f"  Max: {test_df['tlen'].max()} words")

print(f"\nFirst 3 samples:")
for _, row in test_df.head(3).iterrows():
    print(f"  [{row['id']}] {row['input_text'][:80]}")

In [ ]:
# ============================================================
# DATA READY — no DataLoader needed (tokenize per-model in inference)
# ============================================================
# v3: We tokenize inside run_model_on_data() using each model's
# own tokenizer. No need for a shared DataLoader/collate_fn.

print(f"Test data ready: {len(test_df)} samples")
print(f"Batch size: {BATCH_SIZE}")
print(f"Est. batches per model: {math.ceil(len(test_df) / BATCH_SIZE)}")

In [ ]:
# ============================================================
# INFERENCE - v4: FP32 + BF16 autocast, time-budgeted, dual-GPU
# ============================================================

def get_model_device(model):
    """Get the input device for a model."""
    if hasattr(model, 'hf_device_map'):
        first_dev = list(model.hf_device_map.values())[0]
        if isinstance(first_dev, int):
            return torch.device(f'cuda:{first_dev}')
        return torch.device(first_dev)
    return next(model.parameters()).device

def generate_candidates_for_batch(model, tok, input_ids, attn_mask):
    """Generate beam + diverse sample candidates for one batch."""
    B = input_ids.shape[0]
    all_pools = [[] for _ in range(B)]
    
    gen_common = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        repetition_penalty=REPETITION_PENALTY,
        use_cache=True,
    )
    
    # 1. Beam search candidates
    nb = max(NUM_BEAMS, MBR_NUM_BEAM_CANDS)
    beam_out = model.generate(
        input_ids=input_ids, attention_mask=attn_mask,
        do_sample=False, num_beams=nb,
        num_return_sequences=MBR_NUM_BEAM_CANDS,
        length_penalty=LENGTH_PENALTY,
        early_stopping=True,
        **gen_common,
    )
    beam_txt = tok.batch_decode(beam_out, skip_special_tokens=True)
    for i in range(B):
        all_pools[i].extend(beam_txt[i*MBR_NUM_BEAM_CANDS:(i+1)*MBR_NUM_BEAM_CANDS])
    
    # 2. Diverse sampling candidates
    for temp, top_p, count in MBR_SAMPLE_CONFIGS:
        if count <= 0:
            continue
        samp_out = model.generate(
            input_ids=input_ids, attention_mask=attn_mask,
            do_sample=True, num_beams=1,
            temperature=temp, top_p=top_p,
            num_return_sequences=count,
            **gen_common,
        )
        samp_txt = tok.batch_decode(samp_out, skip_special_tokens=True)
        for i in range(B):
            all_pools[i].extend(samp_txt[i*count:(i+1)*count])
    
    return all_pools


def run_model_on_data(model, tok, test_df, desc="Model"):
    """Run one model across all test data, return {id: [candidates]}."""
    pools = {}
    model_dev = get_model_device(model)
    
    for batch_start in tqdm(range(0, len(test_df), BATCH_SIZE), desc=desc):
        # Time check every 50 batches
        if batch_start > 0 and batch_start % (50 * BATCH_SIZE) == 0:
            remaining = time_left()
            if remaining < 1800:
                print(f"  TIME WARNING: {remaining/60:.0f}min left, stopping")
                break
        
        batch_slice = test_df.iloc[batch_start:batch_start+BATCH_SIZE]
        batch_ids = batch_slice['id'].tolist()
        batch_texts = batch_slice['input_text'].tolist()
        
        try:
            encoded = tok(batch_texts, max_length=MAX_LENGTH, padding=True,
                         truncation=True, return_tensors='pt')
            input_ids = encoded.input_ids.to(model_dev)
            attn_mask = encoded.attention_mask.to(model_dev)
            
            with torch.inference_mode():
                # BF16 autocast if supported (T4=FP32, A100=BF16)
                with bf16_autocast_ctx(enabled=USE_BF16):
                    batch_pools = generate_candidates_for_batch(
                        model, tok, input_ids, attn_mask)
            
            for bid, cands in zip(batch_ids, batch_pools):
                pools.setdefault(bid, []).extend(cands)
                
        except Exception as e:
            print(f"  Batch error at {batch_start}: {e}")
            for bid in batch_ids:
                pools.setdefault(bid, [])
        
        if torch.cuda.is_available() and batch_start % (20 * BATCH_SIZE) == 0:
            torch.cuda.empty_cache()
    
    return pools


t0 = time.time()

# --- Phase 1: Primary model ---
print(f"\n{'='*60}")
print(f"Phase 1: Primary model ({model_configs[0]['name']})")
print(f"{'='*60}")

candidate_pools = run_model_on_data(primary_model, primary_tokenizer, test_df, desc='Primary')

elapsed1 = time.time() - t0
n_done = len(candidate_pools)
avg_cands = np.mean([len(v) for v in candidate_pools.values()]) if candidate_pools else 0
print(f"Primary done: {n_done} samples, {avg_cands:.1f} cands/sample, {elapsed1:.0f}s")

# --- Phase 2: Additional models (if any and time permits) ---
if len(model_configs) > 1 and time_left() > 3600:
    # Free primary model to make VRAM room
    free_model(primary_model)
    primary_model = None
    
    for model_idx in range(1, len(model_configs)):
        remaining = time_left()
        est_time = elapsed1 * 1.2
        if remaining < est_time + 1800:
            print(f"\nSkipping model {model_idx+1}: {remaining/60:.0f}min left, need ~{est_time/60:.0f}min")
            break
        
        mcfg = model_configs[model_idx]
        print(f"\n{'='*60}")
        print(f"Phase 2.{model_idx}: {mcfg['name']}")
        print(f"{'='*60}")
        
        try:
            m_tok, m_model = load_model(mcfg['path'])
            
            if hasattr(m_model, 'hf_device_map'):
                print(f"  device_map: {m_model.hf_device_map}")
            
            m_pools = run_model_on_data(m_model, m_tok, test_df, desc=f'Model {model_idx+1}')
            
            for sid, cands in m_pools.items():
                candidate_pools.setdefault(sid, []).extend(cands)
            
            free_model(m_model)
            del m_tok
            
        except Exception as e:
            print(f"  Failed: {e}")

# --- Phase 3: MBR selection + postprocessing ---
print(f"\n{'='*60}")
print(f"Phase 3: MBR ({len(candidate_pools)} samples, utility={MBR_UTILITY})")
print(f"{'='*60}")

results = []
for sid in tqdm(sorted(candidate_pools.keys()), desc='MBR'):
    cands = candidate_pools[sid]
    if USE_MBR and len(cands) > 1:
        best = mbr_pick(cands, utility=MBR_UTILITY)
    elif len(cands) > 0:
        best = cands[0]
    else:
        best = ''
    results.append((sid, best))

# Postprocess
raw_translations = [r[1] for r in results]
cleaned = postprocess_batch(raw_translations)
results = [(results[i][0], cleaned[i]) for i in range(len(results))]

elapsed = time.time() - t0
avg_cands = np.mean([len(v) for v in candidate_pools.values()]) if candidate_pools else 0
print(f"\nDone: {len(results)} translations in {elapsed:.0f}s ({elapsed/max(len(results),1):.1f}s/each)")
print(f"Avg candidates/sample: {avg_cands:.1f}")
print(f"Time remaining: {time_left()/60:.0f} min")

In [ ]:
# ============================================================
# SUBMISSION
# ============================================================

sub = pd.DataFrame(results, columns=['id', 'translation'])
sub = sub.sort_values('id').reset_index(drop=True)
sub['translation'] = sub['translation'].fillna('')

# Validation
empty = sub['translation'].str.strip().eq('').sum()
lengths = sub['translation'].str.len()
print(f"Total: {len(sub)} translations")
print(f"Empty: {empty} ({empty/max(1,len(sub))*100:.1f}%)")
print(f"Lengths: mean={lengths.mean():.0f}, median={lengths.median():.0f}, min={lengths.min()}, max={lengths.max()}")

# Show samples
print(f"\nSamples:")
indices = [0, len(sub)//2, len(sub)-1]
for idx in indices:
    if idx < len(sub):
        row = sub.iloc[idx]
        print(f"  ID {int(row['id']):4d}: {str(row['translation'])[:70]}")

# Save
sub_path = Path(OUTPUT_DIR) / 'submission.csv'
sub.to_csv(sub_path, index=False)
print(f"\nSaved: {sub_path} ({len(sub)} rows)")